In [12]:
using LowLevelFEM, LinearAlgebra

In [13]:
Threads.nthreads()
LinearAlgebra.BLAS.get_num_threads()

2

In [14]:
structured_box_mesh(n=10, order=2)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

Problem("structured_box", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 9261, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [15]:
prob = Problem([mat])
@time K1 = stiffnessMatrix(prob)

  2.565282 seconds (4.92 M allocations: 6.271 GiB, 34.61% gc time)


sparse([1, 2, 3, 49, 50, 51, 79, 80, 81, 82  …  27711, 27721, 27722, 27723, 27775, 27776, 27777, 27781, 27782, 27783], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783], [1754.985754985726, 641.0256410256294, -641.0256410256311, -313.39031339031175, -160.25641025640905, -42.73504273503918, 353.27635327636517, 320.5128205128236, 170.94017094015834, -313.390313390307  …  -820.5128205128224, 4.774847184307873e-12, -2.6542323894318542e-11, 729.3447293447002, -3.1889157980913296e-11, 8.494538406011998e-12, 729.344729344696, 1.0530243343964685e-11, 5.667288860422559e-11, 64182.33618233609], 27783, 27783)

In [16]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))

  0.481399 seconds (108.59 k allocations: 1.507 GiB, 0.45% gc time)


sparse([1, 2, 3, 49, 50, 51, 79, 80, 81, 82  …  27711, 27721, 27722, 27723, 27775, 27776, 27777, 27781, 27782, 27783], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783], [1754.9857549857259, 641.0256410256293, -641.0256410256312, -313.3903133903116, -160.25641025640903, -42.73504273503918, 353.27635327636494, 320.5128205128236, 170.94017094015817, -313.39031339030714  …  -820.5128205128218, 5.030642569181509e-12, -2.6481927761778934e-11, 729.3447293447014, -3.2226665780399344e-11, 8.618883384770015e-12, 729.3447293446904, 1.0032863428932615e-11, 5.6843418860808015e-11, 64182.33618233608], 27783, 27783)

In [17]:
norm(K1.A - K2.A) / norm(K1.A)

2.3907800717529193e-16

In [18]:
structured_rect_mesh(x0=10.0, n=50, order=2)

In [27]:
prob = Problem([mat], type=:AxiSymmetric)

@time K1 = stiffnessMatrix(prob)

  0.157082 seconds (1.11 M allocations: 275.581 MiB, 15.00% gc time)


sparse([1, 2, 9, 10, 107, 108, 699, 700, 799, 800  …  503, 504, 5601, 5602, 20201, 20202, 20397, 20398, 20401, 20402], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402], [6.767206588220727e6, 3.0206010595967714e6, 376292.68629678735, -201545.25177670617, -5.265845710336073e6, 805858.7924758468, -1.1009271415975492e6, 201115.6322682682, 912673.5440149194, -804462.5290736933  …  -4296.195080689155, -2.455189565365407e7, -5.898675847354304e6, -4.2459295993711185e6, 8.323695510625839e-8, -946881.3960378015, 4296.195080269768, -2.455189565365311e7, -2.8032809495925903e-7, 6.798986488703498e7], 20402, 20402)

In [31]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

E = mat.E
ν = mat.ν

r = ScalarField(Pu, "body", (x, y, z)->x)
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]
A2 = [0 0; 1/r 0; 0 0; 0 0]
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu
D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r))

  0.317803 seconds (3.13 M allocations: 323.902 MiB, 16.29% gc time)


sparse([1, 2, 9, 10, 107, 108, 699, 700, 799, 800  …  503, 504, 5601, 5602, 20201, 20202, 20397, 20398, 20401, 20402], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402], [6.767206588220164e6, 3.020601059596482e6, 376292.6862967354, -201545.25177670596, -5.265845710335969e6, 805858.7924756535, -1.1009271415974083e6, 201115.63226817164, 912673.5440146375, -804462.5290733073  …  -4296.195080688939, -2.4551895653654058e7, -5.898675847354108e6, -4.2459295993711185e6, 8.273742224673697e-8, -946881.396037804, 4296.19508026964, -2.4551895653653108e7, -2.7988156148239796e-7, 6.798986488703495e7], 20402, 20402)

In [21]:
norm(K1.A - K2.A) / norm(K1.A)

3.6542243096957084e-14

In [22]:
Threads.nthreads()

4